In [ ]:
%pip install gdown matplotlib
!gdown "1_wv4XFBYoeynXyS8aUb5AH2Cr3fMNbNW"
!gdown "1p50Vn4PGaKlPdPQdooFT0W6GXsz2a0Se"
!gdown "1Civ9LJqf-2mFOqh7o2ozRlRZ9g1jtaWk"
!gdown "1kw27Jzot2aa4VyG7dr_XwzdB42xJsuBw"
%pip install -r requirements.txt

# CyrillicCross: Възстановяване на визуални характеристики между различни езици
![image.png](https://i.ibb.co/spHSbxJF/Cyrillic-Cross-Design-with-Icons.png)


**История**
Екип от изследователи работеше по проекта „CyrillicCross“, когато при тренировъчни сесии се появи софтуерен бъг. Вашият NLI (Natural Language Inference) encoder `RoBERTa-base` бе обучаван да преобразува тези български надписи в embedding-vectors, които после да служат за генериране на изображения със Stable Diffusion. По време на тренировъчни сесии обаче, вследствие софтуерен бъг, dropout остана активиран — половината от елементите на всеки вектор бяха случайно занулени и едновременно circularly shifted (циклично премествани – всеки елемент се измества с определен брой позиции надясно, като елементите, които излизат в края, се връщат в началото). В резултат на това, по време на обучението са генерирани на половина увредени embedding-vector-и.

При тестовите (inference) сесии dropout и circular shift са изключени и embedding-vector-и се създават коректно — тези правилни вектори могат да се използват само за тестване (когато не тренираме projector-а), но докато обучаваме projector-а на задачата за реконструиране, разполагаме единствено с увредените тренировъчни embedding-vector-и.

---

## Изисквания

В рамките на **CyrillicCross**, реализирайте един единствен PyTorch `nn.Module` projector за възстановяване на оцелелите embedding-vector-и:

1. **Вход**

   * `RoBERTa`вектори с размерности `[batch_size, max_seq_length, hidden_size]`, върху, които са приложени droupout и circular shift.

2. **Projector модул**

   * **Коригиране на уврежданията**: Помислете за архитектурно решение, което обръща тези увреждания - dropout и циклично преместване (или привикване към тях?).
   * **Проектиране на векторите**: Проектирайте всеки реконструиран token в CLIP  embedding пространството с размерност `clip_dim`.
   * **Изход**: Помислтете как да извеждате фиксиран по размер тензор `[batch_size, max_seq_length, clip_dim]`.

3. **Ограничения**

   * Не използвайте други библиотеки освен `torch` за архитектурата!
   * Не модифицирайте други файлове; добавяйте код само в секциите маркирани с `#INSERT YOUR CODE HERE`!
   * Нямате право да променяте loss функции или да добавяте такива. Единствено може да се променят хиперпараметрите за трениране!
   * Времето за обучение не трябва да надвишава **20 минути** на GPU A100!

---

## Цел

Възстановяване на CLIP visual embedding–ите, съответстващи на **промпт на български**, изпълнен на английски, като по този начин реставрирате загубените визуални представяния на български език в глобалното CLIP пространство. Метриките за тестване ще бъдат съответно реконструктивна грешка Mean-Squared Error (MSE) и Cosine Similarity базирана грешка (MSE между Cosine similarity между всеки token в prediction-а и в оригиналната поредица). Грешката се смята като Cosine + MSE.

## Изпращане

Изпратете финалното решение като .ipynb файл на notebook-а с ИЗПЪЛНЕНИТЕ КЛЕТКИ, заедно с ЧЕКПОЙНТА, създаден от финалната клетка!!!


In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from transformers import CLIPTextModel, CLIPTokenizer, AutoModel, AutoTokenizer
from datasets import load_dataset
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [ ]:
%run ./tools.py

### Примерен код за circular shift и dropout

In [ ]:

# Примерен embedding-vector за демонстрация
seq_len, hidden_size = 3, 5
x = torch.arange(seq_len * hidden_size, dtype=torch.float32).reshape(seq_len, hidden_size)

# Dropout маска и прилагане
mask = torch.tensor([[1, 0, 1, 1, 0],
                     [0, 1, 0, 1, 1],
                     [1, 1, 0, 0, 1]], dtype=torch.float32)
x_dropped = x * mask

# Circular shift
k = 2
x_shifted = torch.roll(x_dropped, shifts=k, dims=0)

# Визуализация
for data, title in [
    (x, "Оригинален embedding-vector"),
    (x_dropped, "След dropout (прилагане на маска)"),
    (x_shifted, f"След circular shift (k={k})")
]:
    plt.figure()
    plt.imshow(data.numpy(), aspect='auto')
    plt.colorbar()
    plt.title(title)
    plt.xlabel("hidden_size")
    plt.ylabel("seq_len")

plt.show()


### Хиперпараметри

In [ ]:
BATCH_SIZE = 20
LR = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 100
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_id = "sd-legacy/stable-diffusion-v1-5"

### Зараждане на данни

In [ ]:
ds = load_dataset(
    "json",
    data_files={
        "train": "./train.json",
        "validation": "./val.json"
    }
)

dataset_train = ds["train"]
dataset_val = ds["validation"]

### Зареждане на модели

In [ ]:
clip_tokenizer = CLIPTokenizer.from_pretrained(model_id,
    subfolder="tokenizer",        # points to the tokenizer files
    use_fast=True  )
clip_text_encoder = CLIPTextModel.from_pretrained(model_id,
    subfolder="text_encoder" ).to(DEVICE)
clip_text_encoder.eval()
bg_tokenizer = AutoTokenizer.from_pretrained('rmihaylov/roberta-base-nli-stsb-theseus-bg')
bg_encoder = AutoModel.from_pretrained('rmihaylov/roberta-base-nli-stsb-theseus-bg').to(DEVICE)
bg_encoder.eval()
with torch.no_grad():
    dummy_bg = bg_tokenizer(['тест'], return_tensors='pt', padding=True)
    bg_dim = bg_encoder(**{k: v.to(DEVICE) for k, v in dummy_bg.items()}).last_hidden_state.size(-1)
    dummy_en = clip_tokenizer(['test'], return_tensors='pt', padding=True)
    clip_dim = clip_text_encoder(**{k: v.to(DEVICE) for k, v in dummy_en.items()})[0].size(-1)

### Архитектура на projector (Тук се изисква да бъде решението)

In [ ]:
class Projector(nn.Module):
    def __init__(
        self,
        bg_dim: int,
        clip_dim: int,
        num_queries: int = 77,
        nhead: int = 8,
        num_layers: int = 6,
        dim_feedforward: int = 2048,
        dropout: float = 0.1
    ):
        super().__init__()
        self.proj_bg = nn.Linear(bg_dim, clip_dim)

        self.queries = nn.Parameter(torch.empty(num_queries, clip_dim))
        nn.init.normal_(self.queries, mean=0.0, std=0.02)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=clip_dim,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )
        ln = nn.LayerNorm(clip_dim)
        self.transformer = nn.TransformerDecoder(dec_layer, num_layers=num_layers, norm=ln)

        self.proj_out = nn.Linear(clip_dim, clip_dim)

    def forward(self, bg_out: torch.Tensor, bg_mask: torch.Tensor) -> torch.Tensor:
        
        B, L_bg, _ = bg_out.shape

        mem = self.proj_bg(bg_out).permute(1, 0, 2)

        Q = self.queries.size(0)
        q = self.queries.unsqueeze(1).expand(Q, B, -1)


        tgt_key_padding_mask = torch.zeros(B, Q, dtype=torch.bool, device=bg_mask.device)

        memory_key_padding_mask = ~(bg_mask).bool()

        out = self.transformer(
            tgt=q,
            memory=mem,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )

        q_out = out.permute(1, 0, 2)
        refined = self.proj_out(q_out)
        return refined

In [ ]:
random.seed(42)
np.random.seed(42)
torch.cuda.manual_seed(42)
torch.manual_seed(42)

proj = Projector(bg_dim, clip_dim).to(DEVICE)
opt = optim.AdamW(proj.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
train_loader = DataLoader(
    dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda batch: (
        bg_tokenizer([x['bg'].lower() for x in batch], padding=True, truncation=True, return_tensors='pt'),
        clip_tokenizer([x['en'].lower() for x in batch], padding=True, truncation=True, return_tensors='pt')
    )
)
val_loader = DataLoader(
    dataset_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda batch: (
        bg_tokenizer([x['bg'].lower() for x in batch], padding=True, truncation=True, return_tensors='pt'),
        clip_tokenizer([x['en'].lower() for x in batch], padding=True, truncation=True, return_tensors='pt')
    )
)

In [ ]:
%%time
train(proj, bg_encoder, clip_text_encoder, train_loader, val_loader, opt, EPOCHS, DEVICE)

### Валидация

In [ ]:
print(f"Validation: {validate(proj, val_loader, bg_encoder, clip_text_encoder, DEVICE)}")

### Запазване

In [ ]:
proj.eval()
torch.save(proj.state_dict(), "./proj_state_dict.pth")